In [1]:
import pandas as pd
import numpy as np

from tqdm import tqdm
tqdm.pandas()

In [2]:
df = pd.read_csv('songdata.csv')
# df.drop(columns=['link'], inplace=True)

In [3]:
df = df.sample(10000, random_state=42).drop('link', axis=1).reset_index(drop=True)

In [4]:
df.head()

,artist,song,text
0,Wishbone Ash,Right Or Wrong,Like to have you 'round \nWith all the lies t...
1,Aerosmith,This Little Light Of Mine,"This Little Light of Mine (Light of Mine), \n..."
2,Fall Out Boy,"Dance, Dance",She says she's no good with words but I'm wors...
3,Janis Joplin,Easy Rider,"Hey mama, mama, come a look at sister, \nShe'..."
4,Moody Blues,Peak Hour,I see it all through my window it seems. \nNe...


In [5]:
df.shape

(10000, 3)

### Cleaning

In [6]:
df['text'][0]

"Like to have you 'round  \nWith all the lies that you make  \nThe things or darkness and you  \nSome people say, have just a taste  \nRight or wrong, you might get burned  \nWhat you gain is what you learn  \n  \nGot one too many women  \nDon't know quite which way to go  \nThey're all gettin' so expensive  \nWhen they walk by themselves  \nRight or wrong, don't regret  \nWhat you went for is what you get  \n  \nNo point in bitter tears  \nWhen someone else has cut you down  \n'Cause there's a time for leavin'  \nAnd there's a time for stickin' around, hey  \nRight or wrong, you've got to live  \nSo what you collect is what you give\n\n"

In [7]:
df['text'] = df['text'].str.lower().replace(r'[^\w\s]', '').replace(r'\n', ' ', regex=True)

In [8]:
df['text'][0]

"like to have you 'round   with all the lies that you make   the things or darkness and you   some people say, have just a taste   right or wrong, you might get burned   what you gain is what you learn      got one too many women   don't know quite which way to go   they're all gettin' so expensive   when they walk by themselves   right or wrong, don't regret   what you went for is what you get      no point in bitter tears   when someone else has cut you down   'cause there's a time for leavin'   and there's a time for stickin' around, hey   right or wrong, you've got to live   so what you collect is what you give  "

In [9]:
import nltk
from nltk.stem import PorterStemmer

In [10]:
ps = PorterStemmer()

In [11]:
def tokenization(text):
    tokens = nltk.word_tokenize(text)
    stemming = [ps.stem(token) for token in tokens]
    
    for token in tokens:
        stemming.append(ps.stem(token))
    return ' '.join(stemming)

In [12]:
df['text'] = df['text'].progress_apply(lambda x: tokenization(x))

100%|████████████████████████████████████████████████████████████████████████████| 10000/10000 [02:03<00:00, 80.84it/s]


In [13]:
df['text']

0       like to have you 'round with all the lie that ...
1       thi littl light of mine ( light of mine ) , i ...
2       she say she 's no good with word but i 'm wors...
3       hey mama , mama , come a look at sister , she ...
4       i see it all through my window it seem . never...
                              ...                        
9995    you know i 'm hotblood , babi ... get on up an...
9996    you 're the end of the rainbow , my pot of gol...
9997    artist : raw theme lyric song : across the nat...
9998    i wonder who 's sleep in your sheet tonight wh...
9999    when you 're drive down the highway at night a...
Name: text, Length: 10000, dtype: object

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
matrix = tfidf.fit_transform(df['text'])

In [16]:
matrix.shape

(10000, 5000)

In [17]:
similarity = cosine_similarity(matrix)

In [18]:
similarity

array([[1.        , 0.        , 0.03701978, ..., 0.00997877, 0.05445446,
        0.04957703],
       [0.        , 1.        , 0.0215361 , ..., 0.01779531, 0.        ,
        0.03213981],
       [0.03701978, 0.0215361 , 1.        , ..., 0.0159194 , 0.04094327,
        0.01407595],
       ...,
       [0.00997877, 0.01779531, 0.0159194 , ..., 1.        , 0.        ,
        0.00202736],
       [0.05445446, 0.        , 0.04094327, ..., 0.        , 1.        ,
        0.01620602],
       [0.04957703, 0.03213981, 0.01407595, ..., 0.00202736, 0.01620602,
        1.        ]])

In [19]:
similarity.shape

(10000, 10000)

In [20]:
df['song'][0]

'Right Or Wrong'

In [21]:
df[df['song'] == 'Right Or Wrong']

,artist,song,text
0,Wishbone Ash,Right Or Wrong,like to have you 'round with all the lie that ...


In [22]:
# shows the similarity between the song to others
sorted(list(enumerate(similarity[0])), reverse=False, key = lambda x: x[1])

[(1, 0.0),
 (48, 0.0),
 (79, 0.0),
 (110, 0.0),
 (122, 0.0),
 (128, 0.0),
 (203, 0.0),
 (242, 0.0),
 (314, 0.0),
 (346, 0.0),
 (402, 0.0),
 (411, 0.0),
 (448, 0.0),
 (545, 0.0),
 (614, 0.0),
 (697, 0.0),
 (707, 0.0),
 (714, 0.0),
 (715, 0.0),
 (784, 0.0),
 (800, 0.0),
 (803, 0.0),
 (805, 0.0),
 (836, 0.0),
 (878, 0.0),
 (879, 0.0),
 (882, 0.0),
 (962, 0.0),
 (1016, 0.0),
 (1133, 0.0),
 (1225, 0.0),
 (1229, 0.0),
 (1286, 0.0),
 (1289, 0.0),
 (1303, 0.0),
 (1304, 0.0),
 (1319, 0.0),
 (1320, 0.0),
 (1363, 0.0),
 (1368, 0.0),
 (1434, 0.0),
 (1462, 0.0),
 (1484, 0.0),
 (1487, 0.0),
 (1492, 0.0),
 (1528, 0.0),
 (1545, 0.0),
 (1642, 0.0),
 (1775, 0.0),
 (1849, 0.0),
 (1881, 0.0),
 (1919, 0.0),
 (1965, 0.0),
 (2020, 0.0),
 (2053, 0.0),
 (2063, 0.0),
 (2083, 0.0),
 (2111, 0.0),
 (2147, 0.0),
 (2156, 0.0),
 (2157, 0.0),
 (2173, 0.0),
 (2175, 0.0),
 (2197, 0.0),
 (2266, 0.0),
 (2271, 0.0),
 (2287, 0.0),
 (2310, 0.0),
 (2363, 0.0),
 (2380, 0.0),
 (2444, 0.0),
 (2458, 0.0),
 (2490, 0.0),
 (2538, 0.

In [26]:
def recommandation(song):
    if song not in df['song'].values:
        return f"Song '{song}' not found in the dataset."
    
    idx = df[df['song'] == song].index[0]
    
    distances = [(i, sim) for i, sim in enumerate(similarity[idx].flatten()) if i != idx]
    distances = sorted(distances, key=lambda x: x[1], reverse=True)
    
    songs = [df.iloc[i[0]]['song'] for i in distances[1:21]]
    
    return songs


In [27]:
recommandation('Right Or Wrong')

["I'm A Fool To Want You",
 'Where Did We Go Wrong',
 'Too Many Tears',
 'Without You',
 'Do The Right Thing',
 'Right Kind Of Wrong',
 'I Wonder Just Where I Went Wrong',
 'You Got No Right',
 'Wrong Way',
 "I've Been Wrong Before",
 "I'm Gonna Do It Right",
 'Any Way You Want It',
 "Nobody's Wrong",
 'Newborn',
 "I've Got A Right To Cry",
 'Doing All Right',
 'St. Ignatius',
 'One More Time',
 'No Authority',
 'Strong']

In [28]:
import pickle
pickle.dump(df, open('df.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))